#  Chargement, fusion et nettoyage des données brutes (LeBonCoin + BienIci).


In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, count

In [2]:
spark = SparkSession.builder \
    .appName('02_nettoyage_immobilier') \
    .master('local[*]') \
    .config('spark.ui.enabled', 'false') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/16 08:09:57 WARN Utils: Your hostname, codespaces-8b76c5, resolves to a loopback address: 127.0.0.1; using 10.0.1.112 instead (on interface eth0)
26/04/16 08:09:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/16 08:10:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
print('SparkSession cree, version:', spark.version)

SparkSession cree, version: 4.1.1


## Etape 1 — Chargement des deux CSV bruts

In [4]:
RAW = '../data/raw'

In [5]:
df_lbc = spark.read.options(header='true', encoding='UTF-8', multiLine='true', escape='"').csv(RAW + '/leboncoin_raw.csv')

In [6]:
df_bienici = spark.read.options(header='true', encoding='UTF-8', multiLine='true', escape='"').csv(RAW + '/bienici_raw.csv')

In [7]:
print(f'LeBonCoin  : {df_lbc.count()} lignes')

LeBonCoin  : 3870 lignes


In [8]:
print(f'BienIci    : {df_bienici.count()} lignes')

BienIci    : 1908 lignes


- Pour le scrapping du site LeBonCoin nous avons 3870 lignes et 1908 lignes pour BienIci, pour un debut nous allons faire le traitement et tester differents modèles, puis on verra s'il faut rajouter encore plus de lignes.

In [9]:
df_lbc.printSchema()


root
 |-- source: string (nullable = true)
 |-- url_annonce: string (nullable = true)
 |-- type_bien: string (nullable = true)
 |-- prix: string (nullable = true)
 |-- surface: string (nullable = true)
 |-- nb_pieces: string (nullable = true)
 |-- nb_chambres: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- description: string (nullable = true)
 |-- date_scraping: string (nullable = true)



In [10]:
df_bienici.printSchema()


root
 |-- source: string (nullable = true)
 |-- url_annonce: string (nullable = true)
 |-- type_bien: string (nullable = true)
 |-- prix: string (nullable = true)
 |-- surface: string (nullable = true)
 |-- nb_pieces: string (nullable = true)
 |-- nb_chambres: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- description: string (nullable = true)
 |-- date_scraping: string (nullable = true)



- Pour les deux sites, nous avons le même schéma, ce qui va nous permettre de concaténer plus facilement les deux jeux de données pour n’en former qu’un seul. y
- Nous allons ensuite faire plus d’investigation pour repérer les problèmes dans les données et effectuer le nettoyage nécessaire avant de passer aux statistiques, aux visualisations et aux modèles.

## Etape 2 — Fusion des deux sources

In [11]:
df = df_lbc.union(df_bienici)

In [12]:
print(f'Total fusionne : {df.count()} lignes')

Total fusionne : 5778 lignes


## Etape 3 — Vérification et suppression des doublons

- Nous allons commençer par voir s'il y'a des URL en double.

In [13]:
avant_suppression = df.count()

In [14]:
df = df.dropDuplicates(['url_annonce'])
apres_suppression = df.count()

In [15]:
print(f'Avant : {avant_suppression} | Apres : {apres_suppression} | Doublons supprimes : {avant_suppression - apres_suppression}')

Avant : 5778 | Apres : 5763 | Doublons supprimes : 15


In [16]:
df.show(10)

+-------+--------------------+-----------+--------+-------+---------+-----------+--------+-----------+--------------------+-------------+
| source|         url_annonce|  type_bien|    prix|surface|nb_pieces|nb_chambres|   ville|code_postal|         description|date_scraping|
+-------+--------------------+-----------+--------+-------+---------+-----------+--------+-----------+--------------------+-------------+
|bienici|https://www.bieni...|appartement|115000.0|   25.0|      1.0|       NULL|Bordeaux|    33800.0|Bordeaux studio à...|   2026-04-13|
|bienici|https://www.bieni...|appartement|200000.0|   51.0|      1.0|        1.0|Bordeaux|    33000.0|SAINT PAUL\n\nUNI...|   2026-04-13|
|bienici|https://www.bieni...|appartement|135000.0|   28.0|      1.0|       NULL|Bordeaux|    33000.0|Bordeaux Chartron...|   2026-04-13|
|bienici|https://www.bieni...|appartement| 60000.0|   17.0|      1.0|       NULL|Bordeaux|    33000.0|VENTE d'un appart...|   2026-04-13|
|bienici|https://www.bieni...|appa

In [17]:
df.select([
    count(when(col(c).isNull() | (col(c) == ""), c)).alias(c)
    for c in df.columns
]).show()

+------+-----------+---------+----+-------+---------+-----------+-----+-----------+-----------+-------------+
|source|url_annonce|type_bien|prix|surface|nb_pieces|nb_chambres|ville|code_postal|description|date_scraping|
+------+-----------+---------+----+-------+---------+-----------+-----+-----------+-----------+-------------+
|     0|          0|      244|  88|    362|      362|        911|  139|        139|         67|            0|
+------+-----------+---------+----+-------+---------+-----------+-----+-----------+-----------+-------------+



## Etape 4 — Filtre lignes sans prix ou surface

In [18]:
avant_suppression = df.count()
print(f"Avant suppression : {avant_suppression}")

Avant suppression : 5763


In [19]:
df = df.filter(
    F.col("prix").isNotNull() & (F.col("prix") != "") & (F.col("prix") != "None") &
    F.col("surface").isNotNull() & (F.col("surface") != "") & (F.col("surface") != "None")
)

In [20]:
apres_suppression = df.count()
lignes_supprimees = avant_suppression - apres_suppression

In [21]:
print(f"Avant suppression : {avant_suppression} | Après suppression : {apres_suppression} | Supprimées : {lignes_supprimees}")

Avant suppression : 5763 | Après suppression : 5336 | Supprimées : 427


## Etape 5 — Vérification et Correction des villes mal parsées 

In [22]:
valid_villes = ['Paris', 'Lyon', 'Marseille', 'Bordeaux', 'Nantes']

In [23]:
lignes_villes_non_standard = df.filter(~F.col('ville').isin(valid_villes)).count()
print(f'Lignes avec villes non standard : {lignes_villes_non_standard}')

Lignes avec villes non standard : 3


In [24]:
df.filter(~F.col('ville').isin(valid_villes)).select('ville', 'code_postal').show(3, truncate=False)

+---------------------------------+-----------+
|ville                            |code_postal|
+---------------------------------+-----------+
|Villeurbanne                     |69100.0    |
|Part vend proche métro Saint-Just|13004.0    |
|Appartement borely-la plage      |13008.0    |
+---------------------------------+-----------+



- Nous avons trois villes qui n'ont pas la bonne ville, nous allons simplement utiliser le début du code postal pour bien corriger.

In [25]:
df = df.withColumn(
    'ville',
    F.when(
        ~F.col('ville').isin(valid_villes),
        F.when(F.col('code_postal').startswith('75'), 'Paris')
         .when(F.col('code_postal').startswith('69'), 'Lyon')
         .when(F.col('code_postal').startswith('13'), 'Marseille')
         .when(F.col('code_postal').startswith('33'), 'Bordeaux')
         .when(F.col('code_postal').startswith('44'), 'Nantes')
         .otherwise(None)
    ).otherwise(F.col('ville'))
)

In [26]:
lignes_villes_non_standard_apres = df.filter(~F.col('ville').isin(valid_villes)).count()
print(f'Lignes avec villes non standard après correction : {lignes_villes_non_standard_apres}')

Lignes avec villes non standard après correction : 0


## Étape 6 — Changer les types

In [27]:
df = df.withColumn('prix', F.col('prix').cast('double'))

In [28]:
df = df.withColumn('surface', F.col('surface').cast('double'))

In [29]:
df = df.withColumn('nb_chambres', F.col('nb_chambres').cast('double').cast('integer'))

In [30]:
df = df.withColumn('code_postal', F.col('code_postal').cast('double').cast('integer'))

In [31]:
df.printSchema()

root
 |-- source: string (nullable = true)
 |-- url_annonce: string (nullable = true)
 |-- type_bien: string (nullable = true)
 |-- prix: double (nullable = true)
 |-- surface: double (nullable = true)
 |-- nb_pieces: string (nullable = true)
 |-- nb_chambres: integer (nullable = true)
 |-- ville: string (nullable = true)
 |-- code_postal: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- date_scraping: string (nullable = true)



## Étape 7 — Correction du type_bien

In [32]:
df.groupBy('type_bien').count().orderBy(F.desc('count')).show()

+-----------+-----+
|  type_bien|count|
+-----------+-----+
|appartement| 4002|
|     maison|  819|
|     studio|  265|
|     duplex|  184|
|       loft|   42|
|   immeuble|   22|
|    terrain|    2|
+-----------+-----+



In [33]:
df = df.withColumn('type_bien', F.lower(F.col('type_bien')))

In [34]:
print('Valeurs apres correction')
df.groupBy('type_bien').count().orderBy(F.desc('count')).show()

Valeurs apres correction


+-----------+-----+
|  type_bien|count|
+-----------+-----+
|appartement| 4002|
|     maison|  819|
|     studio|  265|
|     duplex|  184|
|       loft|   42|
|   immeuble|   22|
|    terrain|    2|
+-----------+-----+



## Étape 8 — Corriger les descriptions vides

In [35]:
descriptions_null_avant = df.filter(F.col('description').isNull()).count()
print(f'Descriptions null avant : {descriptions_null_avant}')

Descriptions null avant : 44


In [36]:
df = df.withColumn('description', F.coalesce(F.col('description'), F.lit('')))

## Étape 9 — Vérifications finales

In [37]:
df.select([
    F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ''), c)).alias(c)
    for c in df.columns
]).show()

+------+-----------+---------+----+-------+---------+-----------+-----+-----------+-----------+-------------+
|source|url_annonce|type_bien|prix|surface|nb_pieces|nb_chambres|ville|code_postal|description|date_scraping|
+------+-----------+---------+----+-------+---------+-----------+-----+-----------+-----------+-------------+
|     0|          0|        0|   0|      0|        0|        599|    0|          0|         44|            0|
+------+-----------+---------+----+-------+---------+-----------+-----+-----------+-----------+-------------+



- Avant de faire des corrections sur le nombre de chambres nous allons effectuer plus d'analyses.

In [38]:
output_path = '../data/clean'
 
df.write.mode('overwrite').partitionBy('ville').parquet(output_path)
 
print(f'Parquet ecrit dans : {output_path}')
 
# Verification : relire depuis Parquet et comparer le count
df_check = spark.read.parquet(output_path)
print(f'Verification : {df_check.count()} lignes lues depuis Parquet')
spark.stop()


Parquet ecrit dans : ../data/clean
Verification : 5336 lignes lues depuis Parquet
